# Create 7 Disease-Specific Datasets for Single-Label Prediction

Pipeline:
1. Assign each patient to ONE disease (earliest fibrotic visit)
2. Filter case patients (must have prior EHR history)
3. Identify fibrotic ICD columns to remove
4. Build datasets: Cases (y=1) + Controls (y=0) per disease
5. Validate outputs

## Section 0: Config & Imports

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# ── Toggle between sample and full data ──
USE_SAMPLE = True

if USE_SAMPLE:
    FIBROTIC_VISIT_FILE = "fibrotic_gpX_hesinY_Y_visit_sample.csv"
    RAW_EHR_FILE = "gp_icd10_hesin_raw_data_with_ukb_sample.csv"
else:
    FIBROTIC_VISIT_FILE = "fibrotic_gpX_hesinY_Y_visit.csv"
    RAW_EHR_FILE = "gp_icd10_hesin_raw_data_with_ukb.csv"

OUTPUT_DIR = "datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 7 diseases ──
DISEASES = [
    "CKD", "Cardiac_Fibrosis", "MASH", "Pulmonary_fibrosis",
    "SSc_Connective_Tissue", "Crohns_Disease", "Fibrosis_of_Skin"
]

# ── Fibrotic ICD codes per disease (column format: dots removed) ──
FIBROTIC_ICD_CODES = {
    "CKD":                   ["N18", "N181", "N182", "N183", "N184", "N185", "N189"],
    "Cardiac_Fibrosis":      ["I40", "I409", "I420", "I423", "I424", "I425", "I514"],
    "MASH":                  ["K74", "K746", "K75", "K758", "K7581"],
    "Pulmonary_fibrosis":    ["J841", "J8410", "J84112", "J8417", "J848", "J8489", "J849"],
    "SSc_Connective_Tissue": ["M34", "M340", "M341", "M342", "M348", "M349"],
    "Crohns_Disease":        ["K50", "K501", "K508", "K509"],
    "Fibrosis_of_Skin":      ["L905", "L91", "L910"],
}

# Flatten to a single set of all fibrotic codes to remove
ALL_FIBROTIC_CODES = set()
for codes in FIBROTIC_ICD_CODES.values():
    ALL_FIBROTIC_CODES.update(codes)

print(f"Total fibrotic ICD codes to remove: {len(ALL_FIBROTIC_CODES)}")
print(f"Using {'sample' if USE_SAMPLE else 'full'} data")
print(f"Fibrotic visit file: {FIBROTIC_VISIT_FILE}")
print(f"Raw EHR file: {RAW_EHR_FILE}")
print(f"Output directory: {OUTPUT_DIR}/")

## Step 1: Assign Each Patient to ONE Disease

For each patient, find the **earliest** visit in the fibrotic visit file and assign the disease that has a flag of 1. If multiple diseases = 1 at the same earliest visit, **skip** that patient.

In [ ]:
# Load fibrotic visit data
visit_df = pd.read_csv(FIBROTIC_VISIT_FILE, parse_dates=["event_dt"])
print(f"Fibrotic visit file: {len(visit_df)} rows, {visit_df['eid'].nunique()} unique patients")
visit_df.head(10)

In [ ]:
# For each patient, get the earliest visit row
visit_df = visit_df.sort_values(["eid", "event_dt"])
earliest_visits = visit_df.groupby("eid").first().reset_index()

# Identify which disease(s) = 1 at the earliest visit
patient_assignment = {}  # eid -> (disease_name, diagnosis_date)
skipped_multi = []       # patients with multiple diseases at earliest visit

for _, row in earliest_visits.iterrows():
    eid = row["eid"]
    diag_date = row["event_dt"]
    active_diseases = [d for d in DISEASES if row[d] == 1]
    
    if len(active_diseases) == 0:
        # This shouldn't happen (every row in fibrotic visit file should have at least one disease=1)
        print(f"  WARNING: eid={eid} has no disease=1 at earliest visit {diag_date}")
    elif len(active_diseases) > 1:
        # Multiple diseases at the same earliest visit -> skip
        skipped_multi.append(eid)
        print(f"  WARNING: eid={eid} has multiple diseases at earliest visit {diag_date}: {active_diseases} -> SKIPPED")
    else:
        patient_assignment[eid] = (active_diseases[0], diag_date)

print(f"\nPatients assigned to a disease: {len(patient_assignment)}")
print(f"Patients skipped (multi-disease): {len(skipped_multi)}")

# Show assignment counts per disease
from collections import Counter
disease_counts = Counter(v[0] for v in patient_assignment.values())
print("\nAssignment counts per disease:")
for d in DISEASES:
    print(f"  {d}: {disease_counts.get(d, 0)}")

## Step 2: Prior History Filter (Case patients only)

For each case patient, check if their diagnosis date is their **first-ever** visit in the raw EHR data. If yes, exclude them (no prior history for X features). Controls are NOT filtered.

In [ ]:
# Read only eid and event_dt from raw EHR for memory efficiency
raw_dates = pd.read_csv(RAW_EHR_FILE, usecols=["eid", "event_dt"], parse_dates=["event_dt"])
print(f"Raw EHR data: {len(raw_dates)} rows, {raw_dates['eid'].nunique()} unique patients")

# Get each patient's earliest visit date in raw EHR
earliest_raw = raw_dates.groupby("eid")["event_dt"].min().to_dict()

# Filter case patients: exclude those whose diagnosis date == their earliest raw EHR visit
case_patients = {}       # eid -> (disease_name, diagnosis_date)  [filtered]
excluded_no_history = []  # patients excluded because diagnosis is their first EHR visit
excluded_not_in_raw = []  # patients not found in raw EHR at all

for eid, (disease, diag_date) in patient_assignment.items():
    if eid not in earliest_raw:
        excluded_not_in_raw.append(eid)
        continue
    
    patient_earliest = earliest_raw[eid]
    if diag_date <= patient_earliest:
        # Diagnosis date is the first (or before first) EHR visit -> no prior history
        excluded_no_history.append(eid)
    else:
        case_patients[eid] = (disease, diag_date)

print(f"\nCase patients after filtering: {len(case_patients)}")
print(f"Excluded (no prior history): {len(excluded_no_history)}")
print(f"Excluded (not in raw EHR): {len(excluded_not_in_raw)}")

# Show filtered counts per disease
filtered_counts = Counter(v[0] for v in case_patients.values())
print("\nFiltered case counts per disease:")
for d in DISEASES:
    before = disease_counts.get(d, 0)
    after = filtered_counts.get(d, 0)
    print(f"  {d}: {before} -> {after} ({before - after} excluded)")

## Step 3: Identify Fibrotic ICD Columns to Remove

Match the 39 fibrotic ICD codes against raw EHR column names using **exact string matching**. Non-fibrotic codes that share a prefix (e.g., I421, K743) are kept.

In [ ]:
# Read only the header of the raw EHR file
raw_header = pd.read_csv(RAW_EHR_FILE, nrows=0).columns.tolist()
print(f"Total columns in raw EHR: {len(raw_header)}")

# Find which fibrotic codes exist as columns (exact match)
fibrotic_cols_found = ALL_FIBROTIC_CODES.intersection(set(raw_header))
fibrotic_cols_not_found = ALL_FIBROTIC_CODES - fibrotic_cols_found

print(f"\nFibrotic ICD codes found in raw EHR columns ({len(fibrotic_cols_found)}):")
print(f"  {sorted(fibrotic_cols_found)}")
print(f"\nFibrotic ICD codes NOT found ({len(fibrotic_cols_not_found)}):")
print(f"  {sorted(fibrotic_cols_not_found)}")

# Columns to keep: everything except fibrotic codes
cols_to_drop = list(fibrotic_cols_found)
cols_to_keep = [c for c in raw_header if c not in fibrotic_cols_found]
print(f"\nColumns to keep: {len(cols_to_keep)} (dropped {len(cols_to_drop)} fibrotic columns)")

## Step 4: Build 7 Disease Datasets

For each disease:
- **Cases (y=1)**: filtered patients assigned to this disease. X rows = visits before diagnosis. Y row = diagnosis visit.
- **Controls (y=0)**: all patients assigned to OTHER diseases (no prior-history filter). All their raw EHR visits are included.
- Fibrotic ICD columns are excluded via `usecols`.

In [ ]:
# Load raw EHR data with only the columns we need (excluding fibrotic ICD codes)
raw_ehr = pd.read_csv(RAW_EHR_FILE, usecols=cols_to_keep, parse_dates=["event_dt"])
print(f"Raw EHR loaded: {len(raw_ehr)} rows, {raw_ehr['eid'].nunique()} patients, {len(raw_ehr.columns)} columns")

# Build lookup: all patient_assignment eids (before Step 2 filtering) — needed for controls
# Controls = patients assigned to other diseases (including those that failed Step 2 filter for their own disease)
# Actually, controls are patients assigned to ANY other disease from patient_assignment
# They don't need to pass Step 2 filter

# Collect all assigned patients (for control lookup)
all_assigned = patient_assignment  # eid -> (disease, diag_date), includes both filtered and unfiltered

# Summary of who's available
all_assigned_eids_in_raw = set(all_assigned.keys()) & set(raw_ehr["eid"].unique())
print(f"\nAssigned patients found in raw EHR: {len(all_assigned_eids_in_raw)} / {len(all_assigned)}")
print(f"Case patients (passed Step 2 filter): {len(case_patients)}")

In [ ]:
# Build datasets for each disease
dataset_stats = []

for disease in DISEASES:
    print(f"\n{'='*60}")
    print(f"Building dataset for: {disease}")
    print(f"{'='*60}")
    
    # ── Cases (y=1): patients assigned to this disease who passed Step 2 ──
    case_eids = {eid for eid, (d, _) in case_patients.items() if d == disease}
    
    # ── Controls (y=0): patients assigned to ANY OTHER disease ──
    # (from patient_assignment, not case_patients — controls don't need Step 2 filter)
    control_eids = {eid for eid, (d, _) in all_assigned.items() if d != disease}
    
    print(f"  Case patients: {len(case_eids)}")
    print(f"  Control patients: {len(control_eids)}")
    
    rows = []
    
    # ── Process case patients ──
    case_eids_with_data = 0
    for eid in case_eids:
        diag_date = case_patients[eid][1]
        patient_rows = raw_ehr[raw_ehr["eid"] == eid].copy()
        
        if len(patient_rows) == 0:
            continue
        
        case_eids_with_data += 1
        
        # X rows: visits BEFORE diagnosis date
        x_rows = patient_rows[patient_rows["event_dt"] < diag_date].copy()
        x_rows["y_label"] = 1
        x_rows["record_type"] = "x_row"
        
        # Y row: visit ON diagnosis date
        y_rows = patient_rows[patient_rows["event_dt"] == diag_date].copy()
        y_rows["y_label"] = 1
        y_rows["record_type"] = "y_row"
        
        rows.append(x_rows)
        rows.append(y_rows)
    
    # ── Process control patients ──
    control_eids_with_data = 0
    for eid in control_eids:
        patient_rows = raw_ehr[raw_ehr["eid"] == eid].copy()
        
        if len(patient_rows) == 0:
            continue
        
        control_eids_with_data += 1
        patient_rows["y_label"] = 0
        patient_rows["record_type"] = "control"
        rows.append(patient_rows)
    
    # ── Combine and save ──
    if len(rows) > 0:
        dataset = pd.concat(rows, ignore_index=True)
        
        # Reorder columns: eid, event_dt, y_label, record_type, then features
        meta_cols = ["eid", "event_dt", "y_label", "record_type"]
        feature_cols = [c for c in dataset.columns if c not in meta_cols]
        dataset = dataset[meta_cols + feature_cols]
        
        output_path = os.path.join(OUTPUT_DIR, f"dataset_{disease}.csv")
        dataset.to_csv(output_path, index=False)
        
        n_x = len(dataset[dataset["record_type"] == "x_row"])
        n_y = len(dataset[dataset["record_type"] == "y_row"])
        n_ctrl = len(dataset[dataset["record_type"] == "control"])
        
        print(f"  Cases with data in raw EHR: {case_eids_with_data}")
        print(f"  Controls with data in raw EHR: {control_eids_with_data}")
        print(f"  Output rows: {len(dataset)} (x_row={n_x}, y_row={n_y}, control={n_ctrl})")
        print(f"  Output columns: {len(dataset.columns)}")
        print(f"  Saved to: {output_path}")
        
        dataset_stats.append({
            "disease": disease,
            "n_cases": case_eids_with_data,
            "n_controls": control_eids_with_data,
            "n_x_rows": n_x,
            "n_y_rows": n_y,
            "n_control_rows": n_ctrl,
            "n_total_rows": len(dataset),
            "n_columns": len(dataset.columns)
        })
    else:
        print(f"  No data found for this disease (sample data may not have matching patients)")
        dataset_stats.append({
            "disease": disease,
            "n_cases": 0, "n_controls": 0,
            "n_x_rows": 0, "n_y_rows": 0,
            "n_control_rows": 0, "n_total_rows": 0, "n_columns": 0
        })

## Step 5: Validation & Summary

Verify:
1. No fibrotic ICD columns leaked into output
2. No patient appears as both case and control in the same dataset
3. Every case patient has at least 1 x_row and exactly 1 y_row (when data exists)

In [ ]:
# ── Summary table ──
stats_df = pd.DataFrame(dataset_stats)
print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)
print(stats_df.to_string(index=False))
print()

# ── Validation ──
all_passed = True

for disease in DISEASES:
    output_path = os.path.join(OUTPUT_DIR, f"dataset_{disease}.csv")
    if not os.path.exists(output_path):
        print(f"[SKIP] {disease}: no output file (likely no matching patients in sample data)")
        continue
    
    df = pd.read_csv(output_path)
    
    # Check 1: No fibrotic columns
    leaked = set(df.columns) & fibrotic_cols_found
    if leaked:
        print(f"[FAIL] {disease}: fibrotic columns leaked: {leaked}")
        all_passed = False
    else:
        print(f"[PASS] {disease}: no fibrotic columns in output")
    
    # Check 2: No case/control overlap
    case_eids_out = set(df[df["y_label"] == 1]["eid"].unique())
    ctrl_eids_out = set(df[df["y_label"] == 0]["eid"].unique())
    overlap = case_eids_out & ctrl_eids_out
    if overlap:
        print(f"[FAIL] {disease}: {len(overlap)} patients appear as both case and control")
        all_passed = False
    else:
        print(f"[PASS] {disease}: no case/control overlap")
    
    # Check 3: Case patients have x_rows and y_rows
    if len(case_eids_out) > 0:
        for eid in case_eids_out:
            patient_data = df[df["eid"] == eid]
            n_x = len(patient_data[patient_data["record_type"] == "x_row"])
            n_y = len(patient_data[patient_data["record_type"] == "y_row"])
            if n_x == 0:
                print(f"[WARN] {disease}: case eid={eid} has 0 x_rows (no prior visits in raw EHR)")
            if n_y == 0:
                print(f"[WARN] {disease}: case eid={eid} has 0 y_rows (diagnosis visit not found in raw EHR)")

print()
if all_passed:
    print("All validations PASSED!")
else:
    print("Some validations FAILED — check above for details.")